In [66]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
DATA_DIR

WindowsPath('../data')

In [67]:
list(DATA_DIR.iterdir())

[WindowsPath('../data/customer_retention_dataset.parquet'),
 WindowsPath('../data/online_retail_II.xlsx'),
 WindowsPath('../data/online_retail_II_cleaned.parquet'),
 WindowsPath('../data/README.md')]

In [68]:
file_path = DATA_DIR / "online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['Year 2009-2010', 'Year 2010-2011']

In [69]:
for sheet in excel_file.sheet_names:
    sample = pd.read_excel(file_path, sheet_name=sheet, nrows=5)
    print(f"\nSheet: {sheet}")
    display(sample)


Sheet: Year 2009-2010


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



Sheet: Year 2010-2011


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [70]:
frames = []

for sheet in excel_file.sheet_names:
    sheet_df = pd.read_excel(file_path, sheet_name=sheet)
    sheet_df["source_sheet"] = sheet
    frames.append(sheet_df)

raw = pd.concat(frames, ignore_index=True)

raw.shape

(1067371, 9)

In [71]:
display(raw.head())
display(raw.sample(5, random_state=42))

print("Shape:", raw.shape)
print("\nColumns:")
print(raw.columns.tolist())

print("\nData types:")
display(raw.dtypes)


print("\nExact duplicate rows:", raw.duplicated().sum())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
455941,532657,21314,SMALL GLASS HEART TRINKET POT,12,2010-11-14 11:10:00,2.10,"14,562.00",United Kingdom,Year 2009-2010
826291,563214,22383,LUNCH BAG SUKI DESIGN,2,2011-08-14 12:56:00,1.65,"16,370.00",United Kingdom,Year 2010-2011
191636,507597,22561,WOODEN SCHOOL COLOURING SET,12,2010-05-10 13:21:00,1.65,"17,700.00",United Kingdom,Year 2009-2010
25864,491634,21588,RETRO SPOT GIANT TUBE MATCHES,1,2009-12-11 15:40:00,2.55,"17,841.00",United Kingdom,Year 2009-2010
73233,496007,85232B,SET/3 RUSSIAN DOLL STACKING TINS,3,2010-01-28 12:32:00,4.95,"15,203.00",United Kingdom,Year 2009-2010


Shape: (1067371, 9)

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']

Data types:


Invoice                 object
StockCode               object
Description             object
Quantity                 int64
InvoiceDate     datetime64[ns]
Price                  float64
Customer ID            float64
Country                 object
source_sheet            object
dtype: object


Exact duplicate rows: 12133


How many rows and columns are there? 
What time period does the dataset cover?
How many unique customers are present?
What percentage of customer IDs are missing?
How many exact duplicate rows exist?
Are there negative quantities?
Are there zero or negative prices?
How are cancelled invoices represented?
Does each invoice appear across multiple product rows?
What issues must be resolved before defining a valid order?

In [72]:
raw.columns.tolist()


['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country',
 'source_sheet']

In [73]:
INVOICE = "Invoice"
CUSTOMER = "Customer ID"
PRICE = "Price"

print(f"Rows: {len(raw):,}")
print(f"Columns: {raw.shape[1]}")
print(f"Date range: {raw['InvoiceDate'].min()} to {raw['InvoiceDate'].max()}")
print(f"Unique invoices: {raw[INVOICE].nunique():,}")
print(f"Unique customers: {raw[CUSTOMER].nunique():,}")
print(f"Unique products: {raw['StockCode'].nunique():,}")
print(f"Countries: {raw['Country'].nunique():,}")

Rows: 1,067,371
Columns: 9
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique invoices: 53,628
Unique customers: 5,942
Unique products: 5,305
Countries: 43


In [74]:
missing_summary = (
    raw.isna()
       .agg(["sum", "mean"])
       .T
       .rename(columns={"sum": "missing_count", "mean": "missing_rate"})
       .sort_values("missing_rate", ascending=False)
)

missing_summary

,missing_count,missing_rate
Customer ID,"243,007.00",0.23
Description,"4,382.00",0.00
Invoice,0.00,0.00
StockCode,0.00,0.00
Quantity,0.00,0.00
InvoiceDate,0.00,0.00
Price,0.00,0.00
Country,0.00,0.00
source_sheet,0.00,0.00


In [75]:
raw.assign(
    customer_missing=raw[CUSTOMER].isna()
).groupby("customer_missing").agg(
    rows=(INVOICE, "size"),
    invoices=(INVOICE, "nunique"),
    median_quantity=("Quantity", "median"),
    median_price=(PRICE, "median")
)

,rows,invoices,median_quantity,median_price
customer_missing,,,,
False,824364,44876,5.00,1.95
True,243007,8752,1.00,3.29


In [76]:
invoice_text = raw[INVOICE].astype(str).str.strip()

raw["is_cancelled_invoice"] = invoice_text.str.upper().str.startswith("C")

raw["is_cancelled_invoice"].value_counts(dropna=False)

is_cancelled_invoice
False    1047877
True       19494
Name: count, dtype: int64

In [77]:
raw.groupby("is_cancelled_invoice").agg(
    rows=(INVOICE, "size"),
    invoices=(INVOICE, "nunique"),
    customers=(CUSTOMER, "nunique"),
    median_quantity=("Quantity", "median"),
    median_price=(PRICE, "median")
)

,rows,invoices,customers,median_quantity,median_price
is_cancelled_invoice,,,,,
False,1047877,45336,5881,3.00,2.10
True,19494,8292,2572,-2.00,2.95


In [78]:
raw.loc[
    raw["is_cancelled_invoice"],
    [INVOICE, CUSTOMER, "StockCode", "Quantity", PRICE, "InvoiceDate"]
].head(20)

,Invoice,Customer ID,StockCode,Quantity,Price,InvoiceDate
178,C489449,"16,321.00",22087,-12,2.95,2009-12-01 10:33:00
179,C489449,"16,321.00",85206A,-6,1.65,2009-12-01 10:33:00
180,C489449,"16,321.00",21895,-4,4.25,2009-12-01 10:33:00
181,C489449,"16,321.00",21896,-6,2.10,2009-12-01 10:33:00
182,C489449,"16,321.00",22083,-12,2.95,2009-12-01 10:33:00
183,C489449,"16,321.00",21871,-12,1.25,2009-12-01 10:33:00
184,C489449,"16,321.00",84946,-12,1.25,2009-12-01 10:33:00
185,C489449,"16,321.00",84970S,-24,0.85,2009-12-01 10:33:00
186,C489449,"16,321.00",22090,-12,2.95,2009-12-01 10:33:00
196,C489459,"17,592.00",90200A,-3,4.25,2009-12-01 10:44:00


In [79]:
quality_flags = pd.Series({
    "negative_quantity": (raw["Quantity"] < 0).sum(),
    "zero_quantity": (raw["Quantity"] == 0).sum(),
    "negative_price": (raw[PRICE] < 0).sum(),
    "zero_price": (raw[PRICE] == 0).sum(),
    "missing_customer_id": raw[CUSTOMER].isna().sum(),
    "duplicates_including_source_sheet": raw.duplicated().sum(),
    "duplicates_ignoring_source_sheet": raw.duplicated(
        subset=transaction_columns
    ).sum(),
})

quality_flags

negative_quantity                     22950
zero_quantity                             0
negative_price                            5
zero_price                             6202
missing_customer_id                  243007
duplicates_including_source_sheet     12133
duplicates_ignoring_source_sheet      34335
dtype: int64

In [80]:
pd.crosstab(
    raw["Quantity"] < 0,
    raw["is_cancelled_invoice"],
    margins=True
)

is_cancelled_invoice,False,True,All
Quantity,,,
False,1044420,1,1044421
True,3457,19493,22950
All,1047877,19494,1067371


In [81]:
raw.loc[
    (raw["Quantity"] <= 0) | (raw[PRICE] <= 0),
    [INVOICE, CUSTOMER, "StockCode", "Description",
     "Quantity", PRICE, "InvoiceDate"]
].sample(
    n=min(
        20,
        ((raw["Quantity"] <= 0) | (raw[PRICE] <= 0)).sum()
    ),
    random_state=42
)

,Invoice,Customer ID,StockCode,Description,Quantity,Price,InvoiceDate
827532,C563375,"14,911.00",22193,RED DINER WALL CLOCK,-2,8.50,2011-08-16 10:27:00
690256,C550672,"12,457.00",23199,JUMBO BAG APPLES,-100,1.79,2011-04-20 10:10:00
289675,C517565,"13,455.00",21532,DAIRY MAID SUGAR JAM BOWL,-3,2.10,2010-07-30 07:55:00
101075,498930,NaN,72754A,damaged,-62,0.00,2010-02-24 10:24:00
657168,547606,NaN,22173,NaN,-22,0.00,2011-03-24 11:26:00
43639,493180,NaN,22003,NaN,10,0.00,2009-12-22 11:53:00
908445,C569985,"15,365.00",21915,RED HARMONICA IN BOX,-12,1.25,2011-10-06 19:51:00
886079,C568240,"17,172.00",POST,POSTAGE,-1,4.50,2011-09-26 12:04:00
83397,C497161,"14,896.00",21843,RETRO SPOT CAKE STAND,-1,10.95,2010-02-05 16:40:00
559074,C539271,"14,639.00",21843,RED RETROSPOT CAKE STAND,-1,10.95,2010-12-16 15:28:00


In [82]:
duplicate_count_with_source = raw.duplicated().sum()
duplicate_rate_with_source = duplicate_count_with_source / len(raw)

print(
    "Exact duplicates including source_sheet:",
    f"{duplicate_count_with_source:,}"
)

print(
    "Duplicate rate including source_sheet:",
    f"{duplicate_rate_with_source:.2%}"
)

Exact duplicates including source_sheet: 12,133
Duplicate rate including source_sheet: 1.14%


In [83]:
# Columns defining the underlying transaction record.
# source_sheet is excluded because the sheets contain an overlapping period.

transaction_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

duplicate_count = raw.duplicated(
    subset=transaction_columns
).sum()

duplicate_rate = duplicate_count / len(raw)

print(
    "Duplicate records ignoring source_sheet:",
    f"{duplicate_count:,}"
)

print(
    "Duplicate rate ignoring source_sheet:",
    f"{duplicate_rate:.2%}"
)

Duplicate records ignoring source_sheet: 34,335
Duplicate rate ignoring source_sheet: 3.22%


In [84]:
duplicate_rows = raw.duplicated(
    subset=transaction_columns,
    keep=False
)

display(
    raw.loc[duplicate_rows]
       .sort_values(
           ["Invoice", "StockCode", "InvoiceDate", "source_sheet"]
       )
       .head(30)
)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancelled_invoice
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,Year 2009-2010,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,Year 2009-2010,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,"16,329.00",United Kingdom,Year 2009-2010,False


In [85]:
raw.loc[
    raw.duplicated(
        subset=transaction_columns,
        keep=False
    )
].sort_values(
    [
        INVOICE,
        "StockCode",
        "InvoiceDate",
        "source_sheet",
    ]
).head(30)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,is_cancelled_invoice
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,Year 2009-2010,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,Year 2009-2010,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,Year 2009-2010,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,"16,329.00",United Kingdom,Year 2009-2010,False


In [86]:
raw["line_value"] = raw["Quantity"] * raw[PRICE]

raw["line_value"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
)

count   1,067,371.00
mean           18.07
std           292.42
min      -168,469.60
1%             -7.95
5%              0.85
50%             9.90
95%            59.50
99%           180.00
99.9%         829.44
max       168,469.60
Name: line_value, dtype: float64

In [87]:
raw.nlargest(
    20, "line_value"
)[
    [INVOICE, CUSTOMER, "StockCode", "Description",
     "Quantity", PRICE, "line_value", "InvoiceDate"]
]

,Invoice,Customer ID,StockCode,Description,Quantity,Price,line_value,InvoiceDate
1065882,581483,"16,446.00",23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,"168,469.60",2011-12-09 09:15:00
587080,541431,"12,346.00",23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,"77,183.60",2011-01-18 10:01:00
748132,556444,"15,098.00",22502,PICNIC BASKET WICKER 60 PIECES,60,649.50,"38,970.00",2011-06-10 15:28:00
241827,512771,NaN,M,Manual,1,"25,111.09","25,111.09",2010-06-17 16:53:00
432176,530715,"15,838.00",84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,1.69,"15,818.40",2010-11-04 11:36:00
517955,537632,NaN,AMAZONFEE,AMAZON FEE,1,"13,541.33","13,541.33",2010-12-07 15:08:00
540478,537632,NaN,AMAZONFEE,AMAZON FEE,1,"13,541.33","13,541.33",2010-12-07 15:08:00
825443,A563185,NaN,B,Adjust bad debt,1,"11,062.06","11,062.06",2011-08-12 14:50:00
135013,502263,"12,918.00",M,Manual,1,"10,953.50","10,953.50",2010-03-23 15:22:00
135015,502265,NaN,M,Manual,1,"10,953.50","10,953.50",2010-03-23 15:28:00


## Provisional definition of a valid purchase

A valid purchase line must:

1. belong to an invoice that is not marked as cancelled;
2. have a strictly positive quantity;
3. have a strictly positive price;
4. have a non-missing customer ID;
5. not be an exact duplicate row.

A valid order is an invoice containing at least one valid purchase line.

Returns and cancellations are excluded when identifying the first and second
purchases. Their potential use as behavioural information will be considered
separately.

This definition is provisional and will be tested through sensitivity analysis.

In [88]:
cleaned = (
    raw
    .drop_duplicates(
        subset=transaction_columns,
        keep="first"
    )
    .loc[lambda df: ~df["is_cancelled_invoice"]]
    .loc[lambda df: df["Quantity"] > 0]
    .loc[lambda df: df[PRICE] > 0]
    .loc[lambda df: df[CUSTOMER].notna()]
    .copy()
)

cleaned["line_value"] = (
    cleaned["Quantity"] * cleaned[PRICE]
)

print(f"Raw rows: {len(raw):,}")
print(f"Cleaned rows: {len(cleaned):,}")
print(f"Retained: {len(cleaned) / len(raw):.2%}")

Raw rows: 1,067,371
Cleaned rows: 779,425
Retained: 73.02%


In [89]:
assert not cleaned.duplicated(
    subset=transaction_columns
).any()
assert not cleaned["is_cancelled_invoice"].any()
assert cleaned["Quantity"].gt(0).all()
assert cleaned[PRICE].gt(0).all()
assert cleaned[CUSTOMER].notna().all()

In [90]:
for column in cleaned.select_dtypes(include="object").columns:
    print(f"\n{column}")
    print(cleaned[column].map(type).value_counts())


Invoice
Invoice
<class 'int'>    779425
Name: count, dtype: int64

StockCode
StockCode
<class 'int'>    690588
<class 'str'>     88837
Name: count, dtype: int64

Description
Description
<class 'str'>    779425
Name: count, dtype: int64

Country
Country
<class 'str'>    779425
Name: count, dtype: int64

source_sheet
source_sheet
<class 'str'>    779425
Name: count, dtype: int64


In [91]:
string_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Country",
    "source_sheet",
]

for column in string_columns:
    cleaned[column] = cleaned[column].astype("string").str.strip()

In [92]:
cleaned["Customer ID"] = (
    pd.to_numeric(cleaned["Customer ID"], errors="raise")
      .astype("Int64")
      .astype("string")
)

In [93]:
cleaned["Quantity"] = pd.to_numeric(
    cleaned["Quantity"],
    errors="raise"
)

cleaned["Price"] = pd.to_numeric(
    cleaned["Price"],
    errors="raise"
)

cleaned["line_value"] = pd.to_numeric(
    cleaned["line_value"],
    errors="raise"
)

cleaned["InvoiceDate"] = pd.to_datetime(
    cleaned["InvoiceDate"],
    errors="raise"
)

cleaned.dtypes

Invoice                 string[python]
StockCode               string[python]
Description             string[python]
Quantity                         int64
InvoiceDate             datetime64[ns]
Price                          float64
Customer ID             string[python]
Country                 string[python]
source_sheet            string[python]
is_cancelled_invoice              bool
line_value                     float64
dtype: object

In [94]:
output_path = DATA_DIR / "online_retail_II_cleaned.parquet"

cleaned.to_parquet(
    output_path,
    index=False,
    engine="pyarrow"
)

print(f"Saved to: {output_path}")

Saved to: ..\data\online_retail_II_cleaned.parquet


In [95]:
cleaned_check = pd.read_parquet(output_path)

print(cleaned.shape)
print(cleaned_check.shape)
display(cleaned_check.dtypes)

assert cleaned.shape == cleaned_check.shape

(779425, 11)
(779425, 11)


Invoice                 string[python]
StockCode               string[python]
Description             string[python]
Quantity                         int64
InvoiceDate             datetime64[ns]
Price                          float64
Customer ID             string[python]
Country                 string[python]
source_sheet            string[python]
is_cancelled_invoice              bool
line_value                     float64
dtype: object